In [3]:
import pandas as pd

# --- Load ground truth files ---
gt_files = {
    "GT_BDa_Outer": "TSP-BDa_Outer_Random_v2C_All-Proportions.txt",
    "GT_BDa_Outer": "TSP-BDa_Outer_Random_v2C_All-Proportions.txt",
    "GT_HBA": "TSP-HBA_Outer_Random_v2C_All-Proportions.txt",
}

gt_data = {}
for name, path in gt_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    if "TotalCells" in df.columns:
        df.drop(columns=["TotalCells"], inplace=True)
    gt_data[name] = sorted(df.columns.tolist())

# --- Load BayesPrism files ---
bp_files = {
    "BP_BDa_Outer": "BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
    "BP_BDa_Outer": "BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
    "BP_HBA": "BayesPrism-Deconvolutions/TSP-HBA_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
}

bp_data = {}
for name, path in bp_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    bp_data[name] = sorted(df.columns.tolist())

# --- Combine GT + BP side by side ---
cols = {
    "GT_BDa_Outer": gt_data["GT_BDa_Outer"],
    "BP_BDa_Outer": bp_data["BP_BDa_Outer"],
    "GT_BDa_Outer": gt_data["GT_BDa_Outer"],
    "BP_BDa_Outer": bp_data["BP_BDa_Outer"],
    "GT_HBA": gt_data["GT_HBA"],
    "BP_HBA": bp_data["BP_HBA"],
}

# Pad columns to equal length
max_len = max(len(v) for v in cols.values())
for k in cols:
    cols[k] = cols[k] + [""] * (max_len - len(cols[k]))

comparison_df = pd.DataFrame(cols)

# --- Display settings ---
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# --- Print to console ---
print(comparison_df)

# --- Save for manual inspection (Excel-compatible CSV) ---
comparison_df.to_csv("Column_Comparison_GT_vs_BayesPrism.txt", sep="\t", index=False)


                                                                                                                                                          GT_BDa_Outer  \
0                                                                                                                                      OPC/astrocytes/oligodendrocytes   
1                                                                                                                                        acinar cell of salivary gland   
2                                                                           activated cd4-positive, alpha-beta t cell/activated cd8-positive, alpha-beta t cell/t cell   
3                                                                                     adventitial cell/fibroblast/mesenchymal stem cell of adipose tissue/stromal cell   
4                                                                                                                                                     

In [2]:
import pandas as pd

# -------------------------------
# 1) Load ground truth (drop TotalCells, collect sorted cols)
# -------------------------------
gt_files = {
    "GT_BDa_Outer": "TSP-BDa_Outer_Random_v2C_All-Proportions.txt",
    "GT_BDa_Inner": "TSP-BDa_Inner_Random_v2C_All-Proportions.txt",
    "GT_HBA":      "TSP-HBA_Inner_Random_v2C_All-Proportions.txt",
}

gt_data = {}
for name, path in gt_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    if "TotalCells" in df.columns:
        df = df.drop(columns=["TotalCells"])
    gt_data[name] = sorted(df.columns.tolist())

# -------------------------------
# 2) Load BayesPrism files
# -------------------------------
bp_files = {
    "BP_BDa_Outer": "BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
    "BP_BDa_Inner": "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
    "BP_HBA":       "BayesPrism-Deconvolutions/TSP-HBA_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt",
}

# -------------------------------
# 3) Build rename dictionaries from previous comparison file
# -------------------------------
comparison_df = pd.read_csv("Column_Comparison_GT_vs_BayesPrism_Final.txt", sep="\t")

rename_map_bda_outer = dict(
    zip(comparison_df["BP_BDa_Outer"].dropna(), comparison_df["GT_BDa_Outer"].dropna())
)
rename_map_bda_inner = dict(
    zip(comparison_df["BP_BDa_Inner"].dropna(), comparison_df["GT_BDa_Inner"].dropna())
)
rename_map_hba = dict(
    zip(comparison_df["BP_HBA"].dropna(), comparison_df["GT_HBA"].dropna())
)

# -------------------------------
# 4) Apply manual fixes for known mismatches
# -------------------------------
rename_map_hba.update({
    "basal.cell.of.prostate.epithelium": "basal cell of prostate epithelium",
    "basal.cell.medullary.thymic.epithelial.cell": "basal cell/medullary thymic epithelial cell",
})
rename_map_bda_outer.update({
    "endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell":
        "endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell",
    "endothelial.cell.of.arteriole.endothelial.cell.of.venule":
        "endothelial cell of arteriole/endothelial cell of venule",
})
rename_map_bda_inner.update({
    "endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell":
        "endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell",
    "endothelial.cell.of.arteriole.endothelial.cell.of.venule":
        "endothelial cell of arteriole/endothelial cell of venule",
})

rename_maps = {
    "BP_BDa_Outer": rename_map_bda_outer,
    "BP_BDa_Inner": rename_map_bda_inner,
    "BP_HBA":       rename_map_hba,
}

# -------------------------------
# 5) Load BP again, apply rename maps, collect sorted cols
# -------------------------------
bp_data = {}
bp_frames = {}
for name, path in bp_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    df = df.rename(columns=rename_maps[name])
    bp_frames[name] = df
    bp_data[name] = sorted(df.columns.tolist())

# -------------------------------
# 6) Build side-by-side comparison (sorted)
# -------------------------------
cols = {
    "GT_BDa_Outer": gt_data["GT_BDa_Outer"],
    "BP_BDa_Outer": bp_data["BP_BDa_Outer"],
    "GT_BDa_Inner": gt_data["GT_BDa_Inner"],
    "BP_BDa_Inner": bp_data["BP_BDa_Inner"],
    "GT_HBA":       gt_data["GT_HBA"],
    "BP_HBA":       bp_data["BP_HBA"],
}

max_len = max(len(v) for v in cols.values())
for k in cols:
    cols[k] = cols[k] + [""] * (max_len - len(cols[k]))

final_comparison = pd.DataFrame(cols)

# -------------------------------
# 7) Save updated comparison + check matches
# -------------------------------
final_comparison.to_csv("Column_Comparison_GT_vs_BayesPrism_AfterRenaming.txt", sep="\t", index=False)

for pair in [("GT_BDa_Outer", "BP_BDa_Outer"),
             ("GT_BDa_Inner", "BP_BDa_Inner"),
             ("GT_HBA", "BP_HBA")]:
    gt_set = set(gt_data[pair[0]])
    bp_set = set(bp_data[pair[1]])
    common = gt_set & bp_set
    print(f"{pair[1]}: {len(common)} / {len(gt_set)} columns match GT")

# -------------------------------
# 8) Write out renamed BP files (with _renamed.txt suffix)
# -------------------------------
for name, path in bp_files.items():
    out_path = path.replace(".txt", "_renamed.txt")
    bp_frames[name].to_csv(out_path, sep="\t")
    print(f"Saved renamed file: {out_path}")


BP_BDa_Outer: 78 / 78 columns match GT
BP_BDa_Inner: 80 / 80 columns match GT
BP_HBA: 69 / 69 columns match GT
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt
Saved renamed file: BayesPrism-Deconvolutions/TSP-HBA_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt


In [8]:
import pandas as pd

# --- File paths ---
file_original = "BayesPrism-Deconvolutions/TSP-HBA_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt"
file_renamed  = "BayesPrism-Deconvolutions/TSP-HBA_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt"

# --- Load data ---
df_orig = pd.read_csv(file_original, sep="\t", index_col=0)
df_ren  = pd.read_csv(file_renamed,  sep="\t", index_col=0)

# --- Columns to compare ---
cols_orig = ["basal.cell.medullary.thymic.epithelial.cell", "basal.cell.of.prostate.epithelium", "basophil.mast.cell"]
cols_ren  = ["basal cell/medullary thymic epithelial cell", "basal cell of prostate epithelium", "basophil/mast cell"]

# --- Print top 5 values for each ---
print("=== Original file ===")
for c in cols_orig:
    print(f"\nColumn: {c}")
    print(df_orig[c].head(5))

print("\n=== Renamed file ===")
for c in cols_ren:
    print(f"\nColumn: {c}")
    print(df_ren[c].head(5))


=== Original file ===

Column: basal.cell.medullary.thymic.epithelial.cell
TSP27_random_rep2_seed12_10_200.h5ad    0.000024
TSP27_random_rep3_seed4_10_200.h5ad     0.000072
TSP25_random_rep1_seed12_20_400.h5ad    0.014141
TSP27_random_rep5_seed11_20_400.h5ad    0.000069
TSP27_random_rep3_seed7_30_600.h5ad     0.012713
Name: basal.cell.medullary.thymic.epithelial.cell, dtype: float64

Column: basal.cell.of.prostate.epithelium
TSP27_random_rep2_seed12_10_200.h5ad    0.000025
TSP27_random_rep3_seed4_10_200.h5ad     0.000022
TSP25_random_rep1_seed12_20_400.h5ad    0.014205
TSP27_random_rep5_seed11_20_400.h5ad    0.000049
TSP27_random_rep3_seed7_30_600.h5ad     0.000095
Name: basal.cell.of.prostate.epithelium, dtype: float64

Column: basophil.mast.cell
TSP27_random_rep2_seed12_10_200.h5ad    0.000477
TSP27_random_rep3_seed4_10_200.h5ad     0.000103
TSP25_random_rep1_seed12_20_400.h5ad    0.000061
TSP27_random_rep5_seed11_20_400.h5ad    0.000115
TSP27_random_rep3_seed7_30_600.h5ad     0.0000

In [6]:
# --- File paths ---
file_original = "BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt"
file_renamed  = "BayesPrism-Deconvolutions/TSP-BDa_Outer_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt"

# --- Load data ---
df_orig = pd.read_csv(file_original, sep="\t", index_col=0)
df_ren  = pd.read_csv(file_renamed,  sep="\t", index_col=0)

# --- Columns to compare ---
cols_orig = ["endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell", "endothelial.cell.of.arteriole.endothelial.cell.of.venule"]
cols_ren  = ["endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell", "endothelial cell of arteriole/endothelial cell of venule"]

# --- Print top 5 values for each ---
print("=== Original file ===")
for c in cols_orig:
    print(f"\nColumn: {c}")
    print(df_orig[c].head(5))

print("\n=== Renamed file ===")
for c in cols_ren:
    print(f"\nColumn: {c}")
    print(df_ren[c].head(5))

=== Original file ===

Column: endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell
TSP27_random_rep2_seed12_10_200.h5ad    0.015693
TSP27_random_rep3_seed4_10_200.h5ad     0.055072
TSP25_random_rep1_seed12_20_400.h5ad    0.070300
TSP27_random_rep5_seed11_20_400.h5ad    0.000090
TSP27_random_rep3_seed7_30_600.h5ad     0.134026
Name: endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell, dtype: float64

Column: endothelial.cell.of.arteriole.endothelial.cell.of.venule
TSP27_random_rep2_seed12_10_200.h5ad    0.011159
TSP27_random_rep3_seed4_10_200.h5ad     0.034975
TSP25_random_rep1_seed12_20_400.h5ad    0.000318
TSP27_random_rep5_seed11_20_400.h5ad    0.000011
TSP27_random_rep3_seed7_30_600.h5ad     0.000090
Name: endothelial.cell.of.arteriole.endothelial.cell.of.venule, dtype: float64

=== Renamed file ===

Column: endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell
TSP27_random_rep2_seed12_10_200.h5ad    0.015693
TSP2

In [7]:
# --- File paths ---
file_original = "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism.txt"
file_renamed  = "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_All-Counts_BayesPrism_renamed.txt"

# --- Load data ---
df_orig = pd.read_csv(file_original, sep="\t", index_col=0)
df_ren  = pd.read_csv(file_renamed,  sep="\t", index_col=0)

# --- Columns to compare ---
cols_orig = ["endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell", "endothelial.cell.of.arteriole.endothelial.cell.of.venule"]
cols_ren  = ["endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell", "endothelial cell of arteriole/endothelial cell of venule"]

# --- Print top 5 values for each ---
print("=== Original file ===")
for c in cols_orig:
    print(f"\nColumn: {c}")
    print(df_orig[c].head(5))

print("\n=== Renamed file ===")
for c in cols_ren:
    print(f"\nColumn: {c}")
    print(df_ren[c].head(5))

=== Original file ===

Column: endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell
TSP27_random_rep2_seed12_10_200.h5ad    0.012750
TSP27_random_rep3_seed4_10_200.h5ad     0.050775
TSP25_random_rep1_seed12_20_400.h5ad    0.069744
TSP27_random_rep5_seed11_20_400.h5ad    0.000070
TSP27_random_rep3_seed7_30_600.h5ad     0.129849
Name: endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell, dtype: float64

Column: endothelial.cell.of.arteriole.endothelial.cell.of.venule
TSP27_random_rep2_seed12_10_200.h5ad    0.016254
TSP27_random_rep3_seed4_10_200.h5ad     0.040470
TSP25_random_rep1_seed12_20_400.h5ad    0.000094
TSP27_random_rep5_seed11_20_400.h5ad    0.000017
TSP27_random_rep3_seed7_30_600.h5ad     0.000126
Name: endothelial.cell.of.arteriole.endothelial.cell.of.venule, dtype: float64

=== Renamed file ===

Column: endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell
TSP27_random_rep2_seed12_10_200.h5ad    0.012750
TSP2